In [1]:
import os
import sys

# Project root
project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root added:")
print(project_root)

Project root added:
/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance


In [2]:
from config.settings import *

print("Settings loaded successfully")
print("Raw data path:", DATA_RAW_PATH)
print("Processed data path:", DATA_PROCESSED_PATH)

Settings loaded successfully
Raw data path: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/raw
Processed data path: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/processed


In [3]:
from pathlib import Path

RAW_DIR = Path(DATA_RAW_PATH) / "webpages" / "komeil_roudi"

article_files = sorted(RAW_DIR.glob("*/article.json"))

print(f"Found {len(article_files)} articles.\n")

for file in article_files[:5]:
    print(file)

Found 76 articles.

/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/raw/webpages/komeil_roudi/00D9A87F/article.json
/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/raw/webpages/komeil_roudi/0A050EE9/article.json
/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/raw/webpages/komeil_roudi/11140315/article.json
/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/raw/webpages/komeil_roudi/14509512/article.json
/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/raw/webpages/komeil_roudi/1699F2E4/article.json


In [4]:
import json

with open(article_files[0], "r", encoding="utf-8") as f:
    article = json.load(f)

print("Keys:")
print(article.keys())

print("\nTitle:")
print(article["title"])

print("\nURL:")
print(article["url"])

print("\nContent Preview:")
print(article["content"][:1000])

Keys:
dict_keys(['url', 'title', 'content'])

Title:
چطور شغل‌مان را توسعه دهیم؟

URL:
https://www.fintelligence.ir/Blog/Detail/00D9A87F/%DA%86%D8%B7%D9%88%D8%B1-%D8%B4%D8%BA%D9%84%D9%85%D8%A7%D9%86-%D8%B1%D8%A7-%D8%AA%D9%88%D8%B3%D8%B9%D9%87-%D8%AF%D9%87%DB%8C%D9%85

Content Preview:



In [5]:
import re


def preprocess_text(text: str) -> str:
    """
    Basic text preprocessing for Persian articles.
    """

    if not text:
        return ""

    # Remove invisible characters
    text = text.replace("\u200c", "\u200c")  # Keep ZWNJ
    text = text.replace("\ufeff", "")
    text = text.replace("\u200f", "")
    text = text.replace("\u200e", "")

    # Normalize newlines
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Remove extra spaces
    text = re.sub(r"[ \t]+", " ", text)

    # Remove extra blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [6]:
text = article["content"]

clean_text = preprocess_text(text)

print("Original length :", len(text))
print("Processed length:", len(clean_text))

print("\nPreview:")
print(clean_text[:1000])

Original length : 0
Processed length: 0

Preview:



In [7]:
import json

for file in article_files:
    with open(file, "r", encoding="utf-8") as f:
        article = json.load(f)

    if article["content"].strip():
        print("Found article:")
        print(file)
        print()

        print("Title:")
        print(article["title"])
        print()

        print("Content length:", len(article["content"]))
        print()

        processed = preprocess_text(article["content"])

        print("Processed preview:\n")
        print(processed[:1500])

        break

Found article:
/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/raw/webpages/komeil_roudi/0A050EE9/article.json

Title:
سه میزان اصلی در ارزیابی فیلم‌های سینمایی از منظر سواد مالی

Content length: 5731

Processed preview:

هر فیلمی به موضوعی می‌پردازد و پرداختن به آن را در قالب گره اصلی و فرعی و اشارات می‌گنجاند. کارگردان به ویژه در طول فیلم فریاد می‌زند که «من پیامی با پرداختن به این موضوع دارم». گاه زبان فیلم نارساست و گاه گوش با واژه‌ها ناآشنا؛ اما سینما زمانی سینماست که پیامی رسا به واژه‌هایی آشنا بیان شود. این هر دو به ترتیب به این معناست که اول، فیلم بیان سینمایی دارد و دوم، مخاطب‌فهم و مخاطب‌پسند است. زمانی که فیلم بیان سینمایی ندارد، از سینما فاصله می‌گیرد؛ هر چند در سینما به نمایش درآید. سینما هنر هفتم و به بیان فنی‌تر، ترکیب شش هنر است. بیان سینمایی یعنی هر یک از آن شش هنر در جای خود و به اندازهٔ خود و با ترکیب مناسب، حامل پیام کارگردان است. با این وصف، در ارزیابی فیلم نمی‌شود بیان سینمایی را نادیده گرفت. مخاطب‌فهمی و مخاطب‌پسندی به معنای عوام‌زدگی یا ت

In [8]:
import json
from pathlib import Path
from tqdm import tqdm

# مسیر خروجی
OUTPUT_DIR = Path(DATA_PROCESSED_PATH) / "komeil_roudi"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

processed_count = 0

for file in tqdm(article_files):

    with open(file, "r", encoding="utf-8") as f:
        article = json.load(f)

    processed_article = {
        "url": article["url"],
        "title": preprocess_text(article["title"]),
        "content": preprocess_text(article["content"])
    }

    article_id = file.parent.name

    save_dir = OUTPUT_DIR / article_id
    save_dir.mkdir(exist_ok=True)

    with open(save_dir / "article.json", "w", encoding="utf-8") as f:
        json.dump(processed_article, f, ensure_ascii=False, indent=2)

    processed_count += 1

print("=" * 50)
print(f"Processed articles: {processed_count}")
print(f"Saved to: {OUTPUT_DIR}")

100%|██████████| 76/76 [00:00<00:00, 493.97it/s]

Processed articles: 76
Saved to: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/processed/komeil_roudi


In [9]:
import json
from pathlib import Path

processed_dir = Path(DATA_PROCESSED_PATH) / "komeil_roudi"

files = sorted(processed_dir.glob("*/article.json"))

print("Processed files:", len(files))

with open(files[0], "r", encoding="utf-8") as f:
    article = json.load(f)

print("\nKeys:")
print(article.keys())

print("\nTitle:")
print(article["title"])

print("\nContent length:")
print(len(article["content"]))

print("\nPreview:")
print(article["content"][:800])

Processed files: 76

Keys:
dict_keys(['url', 'title', 'content'])

Title:
چطور شغل‌مان را توسعه دهیم؟

Content length:
0

Preview:



In [10]:
import json
from pathlib import Path

processed_dir = Path(DATA_PROCESSED_PATH) / "komeil_roudi"

for file in sorted(processed_dir.glob("*/article.json")):

    with open(file, "r", encoding="utf-8") as f:
        article = json.load(f)

    if article["content"]:

        print("ID:", file.parent.name)
        print("Title:", article["title"])
        print("Length:", len(article["content"]))
        print()
        print(article["content"][:800])

        break

ID: 0A050EE9
Title: سه میزان اصلی در ارزیابی فیلم‌های سینمایی از منظر سواد مالی
Length: 5731

هر فیلمی به موضوعی می‌پردازد و پرداختن به آن را در قالب گره اصلی و فرعی و اشارات می‌گنجاند. کارگردان به ویژه در طول فیلم فریاد می‌زند که «من پیامی با پرداختن به این موضوع دارم». گاه زبان فیلم نارساست و گاه گوش با واژه‌ها ناآشنا؛ اما سینما زمانی سینماست که پیامی رسا به واژه‌هایی آشنا بیان شود. این هر دو به ترتیب به این معناست که اول، فیلم بیان سینمایی دارد و دوم، مخاطب‌فهم و مخاطب‌پسند است. زمانی که فیلم بیان سینمایی ندارد، از سینما فاصله می‌گیرد؛ هر چند در سینما به نمایش درآید. سینما هنر هفتم و به بیان فنی‌تر، ترکیب شش هنر است. بیان سینمایی یعنی هر یک از آن شش هنر در جای خود و به اندازهٔ خود و با ترکیب مناسب، حامل پیام کارگردان است. با این وصف، در ارزیابی فیلم نمی‌شود بیان سینمایی را نادیده گرفت. مخاطب‌فهمی و مخاطب‌پسندی به معنای عوام‌زدگی یا توده‌گرایی نیست. سینمایی سینماست که فرد را از خ
